In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatMessagePromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv
load_dotenv()

C:\Users\Deepam\AppData\Local\Temp\ipykernel_3924\2687492296.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

# Make sure GOOGLE_API_KEY is set in your .env file

loader = PyPDFLoader(r"D:\vector_databse\projeect for rag pipeline\dataset\hr_policy_manual.pdf")
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ".", ","],
    chunk_overlap=100,
)
chunks = splitter.split_documents(pages)

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = FAISS.from_documents(chunks, embeddings)

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
ret = vectorstore.as_retriever(seach_kwargs={"k":3})
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

In [4]:
ret.invoke("what is the notice period for senior managers?")

[Document(id='91fe149b-954a-4008-83a3-648e3ef7b6f9', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-31T19:53:57+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-31T19:53:57+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'D:\\vector_databse\\projeect for rag pipeline\\dataset\\hr_policy_manual.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='NovaTech Solutions Pvt. Ltd.\nEmployee HR Policy Manual — FY 2024-25 | Version 3.2 | Confidential\nSection 1: Employment Terms\n1.1 Probation Period\nAll new employees at NovaTech Solutions are subject to a probation period of 6 months from their\ndate of joining. During the probation period, either party may terminate employment with a notice\nperiod of 15 days. Confirmation is subject to a satisfactory performance review by the reporting\nmanager and HR Business Partner.\n1.2 Notice Period\nPost-co

In [5]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    "answer using context only. \n {context}\nQuestion: {question}"
)

def fmt(docs):
    return "\n\n".join([doc.page_content for doc in docs])

chain = (
    {"context": ret | fmt, "question" : RunnablePassthrough()}
    |prompt | llm | StrOutputParser()
)


In [6]:
def ask(q):
    return chain.invoke(q), ret.invoke(q)

In [9]:
ans, src = ask("What is the notice peroid for senior mangers?")

ChatGoogleGenerativeAIError: Error calling model ' gemini-1.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': '* GenerateContentRequest.model: unexpected model name format\n', 'status': 'INVALID_ARGUMENT'}}